# 🌩️ CloudSRE v2 — Custom Unsloth GRPO Training
**OpenEnv India Hackathon 2026 Finale | HF Space | GitHub**

Train our 1.5B parameter SRE agent using a **Custom Unsloth GRPO** implementation. The agent learns to diagnose and fix cascading microservice outages across a 16-service asynchronous environment, outsmarting a 72B Adversarial Designer.

| Component | Detail |
| :--- | :--- |
| **Environment** | HF Space (16-service asynchronous simulated environment) |
| **Training** | This Colab notebook (T4 / A100 GPU) |
| **Model** | `Qwen/Qwen2.5-1.5B` + Unsloth LoRA |
| **Framework** | Custom Unsloth GRPO (Group Relative Policy Optimization) |

---

## 1. Install Dependencies & Clone Repository
Install the highly optimized Unsloth library. We also clone the environment repository to access the reward functions and custom GRPO training script.

In [ ]:
!pip install -q "torch>=2.4.0,<2.11.0" "torchvision>=0.19.0,<0.26.0" 2>/dev/null
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null
!pip install -q unsloth_zoo accelerate bitsandbytes xformers triton httpx wandb matplotlib seaborn 2>/dev/null

!rm -rf CloudSRE
!git clone https://github.com/Harikishanth/CloudSRE.git
%cd CloudSRE


## 2. Configuration & Secrets
Set the HF Space URL, model, and hyperparameters. Add `HF_TOKEN` to Colab Secrets (the 🔑 key icon in the left sidebar).

In [ ]:
import os

# --- HuggingFace token ---
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not found. Set it in Colab Secrets or as env var.")

# --- Environment server (HF Space) ---
ENV_URL = "https://dardrax-cloudsre-environment.hf.space"

# --- Model ---
MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

# --- Training hyperparameters ---
NUM_EPISODES = 5 
NUM_GENERATIONS = 4
CURRICULUM = "warmup,single_fault,cascade,multi_cascade,adversarial"

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")


## 3. Smoke Test — Verify Environment Connectivity
Connect to the HuggingFace Space, reset the 16-service environment, and run a healthcheck command to confirm the round-trip works.

In [ ]:
import httpx

print(f"Connecting to {ENV_URL} ...")

client = httpx.Client(base_url=ENV_URL, timeout=30.0)

try:
    # 1. Reset Environment
    reset_res = client.post("/reset").json()
    print("✅ Connected and Reset successfully!")
    print(f"Alert Context:\n{reset_res['observation']['context'][:300]}...\n")
    
    # 2. Run Smoke Test Command
    step_payload = {"action": {"command": "healthcheck api-gateway"}}
    step_res = client.post("/step", json=step_payload).json()
    
    print(f"--- Smoke Test Step (Command: healthcheck api-gateway) ---")
    print(f"Reward: {step_res['reward']:.2f}")
    print(f"Output:\n{step_res['observation']['command_output']}")
    print("\n✅ Smoke test passed. Environment is ready for training.")
except Exception as e:
    print(f"❌ Connection failed: {e}")


## 4. Run Custom Unsloth GRPO Training
We execute our custom GRPO training script. It handles group-relative advantage computation and policy gradient clipping purely through Unsloth's highly optimized forward/backward passes.

*(Note: We pass `--no-wandb` here so the notebook doesn't hang waiting for a login token!)*

In [ ]:
!python training_scripts/train_grpo.py \
    --env-url {ENV_URL} \
    --model-id {MODEL_ID} \
    --curriculum {CURRICULUM} \
    --episodes-per-tier {NUM_EPISODES} \
    --group-size {NUM_GENERATIONS} \
    --max-turns 5 \
    --lr 5e-6 \
    --output-dir "./cloudsre-grpo-demo" \
    --no-wandb


## 5. Visualize Reward Curve
Use matplotlib and seaborn to visualize the training breakthrough.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

log_path = "data/grpo_training_log.json"

try:
    import json
    with open(log_path) as f:
        log = json.load(f)

    rewards = [ep["reward"] for ep in log["per_episode"]]
    episodes = list(range(1, len(rewards) + 1))

    plt.figure(figsize=(10, 5))
    sns.lineplot(x=episodes, y=rewards, marker='o', color='#3498db', linewidth=2)
    plt.title('Unsloth GRPO Training Reward Curve', fontsize=14, fontweight='bold')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.grid(True, alpha=0.3)
    plt.show()
except Exception as e:
    print(f"Could not plot (training may have been interrupted): {e}")
